# PPO / DDQN training on Colab CPU

Runs `Strategy 2`'s PPO/DDQN training pipeline on a Colab runtime instead of a local laptop, so training doesn't tie up your machine and can run longer/in parallel with other sessions.

Read `Strategy 2/CLAUDE.md` first, especially: heavy training belongs here (Colab), not local; never train PPO/DDQN on ASU's chosen actions (playing against ASU as an opponent seat is fine); report win rates with opponent identity, seat balance, and round cap, never a bare percentage.

**Runtime**: Runtime menu -> Change runtime type -> CPU. This pipeline is CPU-only; a GPU runtime buys nothing here and just burns your GPU quota.

**Session limits**: free Colab disconnects after ~90 min idle and has a hard ceiling per session (~12h). This notebook checkpoints to Drive every `--checkpoint-every` games and supports `--resume`, so a disconnect only costs the games since the last checkpoint -- rerun the training cell with `--resume` after reconnecting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DeepRL_Monopoly'
!mkdir -p "$DRIVE_ROOT/artifacts"

## Get the code

Clones the repo fresh each session (Colab's local disk is ephemeral). Requires the branch with this work pushed to `origin` first -- if you get a 404 or an old tree, push from your laptop and rerun this cell.

In [ ]:
REPO_URL = 'https://github.com/Darkosxl/DeepRL_Monopoly.git'
BRANCH = 'main'

!rm -rf /content/DeepRL_Monopoly
!git clone --branch $BRANCH --depth 1 $REPO_URL /content/DeepRL_Monopoly
%cd "/content/DeepRL_Monopoly/Strategy 2"
!ls

In [ ]:
# Colab ships torch preinstalled; this just confirms the CPU build works
# and numpy is present. No CUDA wheel needed since this pipeline is CPU-only.
!pip install -q --upgrade numpy
import torch, numpy
print('torch', torch.__version__, 'cuda available (unused here):', torch.cuda.is_available())
print('numpy', numpy.__version__)

## Train PPO (hybrid)

`--opponent-epsilon` / `--opponent-threshold-jitter` add noise to the three fixed opponents so the learner doesn't overfit to one deterministic trio; `--held-out-eval-games` probes against TheRailBaron, which is never in the training pool, as an overfitting check. Raise `--games` once a short run shows the win rate moving off 0%.

In [ ]:
PPO_OUT = f'{DRIVE_ROOT}/artifacts/ppo_hybrid_colab.pt'

!python tools/train_and_save.py \
  --algo ppo --hybrid \
  --games 2000 \
  --device cpu \
  --seed 42 \
  --checkpoint-every 100 \
  --opponent-epsilon 0.1 \
  --opponent-threshold-jitter 0.2 \
  --held-out-eval-games 20 \
  --out "$PPO_OUT"

Resume after a disconnect (same `--out`, add `--resume`):

In [ ]:
!python tools/train_and_save.py \
  --algo ppo --hybrid --resume \
  --games 5000 \
  --device cpu --seed 42 --checkpoint-every 100 \
  --opponent-epsilon 0.1 --opponent-threshold-jitter 0.2 --held-out-eval-games 20 \
  --out "$PPO_OUT"

## Train DDQN (hybrid) -- with the diagnosed short-run fix

Default DDQN `lr=1e-5` and `target_update_freq=500 games` are tuned for a 10,000-game paper run; a 1000-game run at those defaults stayed at 0% win rate with correctly-signed rewards the whole time -- the network wasn't absorbing credit fast enough for the run length (see `Strategy 2/CLAUDE.md`, "Known-hard problem"). `--lr` and `--target-update-freq-steps` below compensate for a shorter budget; `--epsilon-decay` reaches the exploration floor sooner too.

In [ ]:
DDQN_OUT = f'{DRIVE_ROOT}/artifacts/ddqn_hybrid_colab.pt'

!python tools/train_and_save.py \
  --algo ddqn --hybrid \
  --games 2000 \
  --device cpu \
  --seed 42 \
  --checkpoint-every 100 \
  --lr 1e-4 \
  --target-update-freq-steps 2000 \
  --epsilon-decay 0.9985 \
  --opponent-epsilon 0.1 \
  --opponent-threshold-jitter 0.2 \
  --held-out-eval-games 20 \
  --out "$DDQN_OUT"

## Check the result

Report format follows the CLAUDE.md rule: win rate needs its opponent identity and setup, not a bare percentage.

In [ ]:
import json

for label, out in [('PPO', PPO_OUT), ('DDQN', DDQN_OUT)]:
    history_path = out.rsplit('.', 1)[0] + '_history.json'
    try:
        with open(history_path) as f:
            history = json.load(f)
    except FileNotFoundError:
        print(f'{label}: no history yet ({history_path})')
        continue
    win_rates = history.get('win_rates') or []
    held_out = history.get('held_out_win_rate') or []
    print(f"{label}: games_completed={history.get('games_completed')} "
          f"final_win_rate_vs_pool={win_rates[-1] if win_rates else None} "
          f"best_win_rate_vs_pool={max(win_rates) if win_rates else None} "
          f"final_held_out_vs_RailBaron={held_out[-1] if held_out else None}")

Checkpoints and history already live under `$DRIVE_ROOT/artifacts` on Drive (that's the `--out` path above), so they survive the Colab VM being recycled. To pull a checkpoint back to your laptop, download it from Drive directly -- don't route it back through git; `artifacts/` is gitignored on purpose.